# Level 2 — Vision Transformers (ViT-S/16, 선택적으로 Swin-Tiny)

**목표**: ViT (와 선택적으로 Swin-T) 를 직접 구현하고 Multi-task 로 연결하여 Level 1 의 CNN 들과 비교합니다.

**Pretrained 가중치**: ImageNet `.pth` 파일을 본인이 구현한 모델의 `state_dict` 에 로드하는 것은 허용됩니다. 출처를 명시하세요. **`timm` / `torchvision.models` import 는 금지** 입니다.

In [ ]:
import os
import sys

# 1. 코랩 환경에서 레포지토리가 클론되지 않은 경우에만 Clone 진행
repo_name = "2026-HYU-AUE8088-PA2"
if not os.path.exists(f"/content/{repo_name}"):
    !git clone https://github.com/dahye411/2026-HYU-AUE8088-PA2.git

# 2. 작업 디렉토리를 레포지토리의 최상단(Root)으로 변경
%cd /content/{repo_name}

%load_ext autoreload
%autoreload 2

# 의존성 설치 (이미 설치된 패키지는 빠르게 skip)
!pip install -q -r requirements.txt

SyntaxError: invalid syntax (2096614278.py, line 16)

In [2]:
import torch
from torch import nn
from torch.utils.data import DataLoader

from src.utils.seed import set_seed, seed_worker
from src.utils.transforms import train_transform, eval_transform
from src.utils.trainer import MultiTaskTrainer, TrainConfig
from src.utils.wandb_logger import WandbLogger
from src.utils.metrics import collect_predictions, confusion_matrices, CLASS_NAMES
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES
from src.models.vit import vit_small_patch16_224
# from src.models.swin import SwinTiny  # 선택 사항

SEED = 42
set_seed(SEED, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
import wandb; wandb.login()   # API key 입력

# wandb 설정 — 비활성화하려면 None
WANDB_PROJECT = "aue8088-pa2"
WANDB_TAGS    = ["level2"]

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sere411 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
DATA_ROOT = "../data/set_a"
BATCH = 64

# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
# ../data/set_a 가 없으면 zip 을 받아 상위 폴더에 압축 해제 → ../data/set_a, ../data/set_b 생성.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "../aue8088_pa2_data.zip"
EXTRACT_TO = ".."   # zip 내부 최상위가 data/ 이므로 상위 폴더에 풀면 ../data/... 가 됨

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------

train_ds = BDDAttrDataset(DATA_ROOT, "train", transform=train_transform())
val_ds   = BDDAttrDataset(DATA_ROOT, "val",   transform=eval_transform())

g = torch.Generator(); g.manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, worker_init_fn=seed_worker, generator=g, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

데이터셋 zip 다운로드 중...


Downloading...
From (original): https://drive.google.com/uc?id=1L7YC70QlO87aIbE5lbtQ94HUINJijBKK
From (redirected): https://drive.google.com/uc?id=1L7YC70QlO87aIbE5lbtQ94HUINJijBKK&confirm=t&uuid=88e71e53-0749-41bf-80a9-5f45574a008f
To: /content/aue8088_pa2_data.zip
100%|██████████| 243M/243M [00:01<00:00, 235MB/s] 


압축 해제 중...
완료 → ../data/set_a


In [ ]:
# 선택 사항: 본인 ViT 구현체에 ImageNet pretrained 가중치를 로드하는 절차
#
# 진행 방식:
#   1) 공개된 ViT-S/16 체크포인트 (.pth) 를 다운로드.
#   2) 모델 인스턴스 생성:  model = vit_small_patch16_224()
#   3) 키 매핑 후 로드:
#        pre = torch.load('vit_s16.pth')
#        missing, unexpected = model.load_state_dict(remap(pre), strict=False)
#        # Multi-task head 는 task 종속이므로 random init 유지.
#
# 사용한 체크포인트 출처와 매칭된 키 개수를 리포트에 기재하세요.
USE_PRETRAINED = True
PRETRAINED_PATH = "./pretrained_checkpoints/deit_small_patch16_224-cd65a155.pth"
PRETRAINED_SOURCE = "https://huggingface.co/facebook/deit-small-patch16-224"

model = vit_small_patch16_224()

# 현재 ViT 구현체와 이름 및 shape이 일치하는 pretrained tensor만 반환
def remap(pre):
    if isinstance(pre, dict) and "state_dict" in pre:
        pre = pre["state_dict"]
    elif isinstance(pre, dict) and "model" in pre:
        pre = pre["model"]

    model_state = model.state_dict()
    mapped = {}

    for k, v in pre.items():
        if not torch.is_tensor(v):
            continue

        # 저장 방식에 따라 붙는 wrapper prefix를 제거
        k = k.removeprefix("module.")
        k = k.removeprefix("backbone.")
        k = k.removeprefix("model.")

        # Multi-task head는 현재 task에 맞춰 새로 학습해야 하므로 제외
        if k.startswith("head."):
            continue

        if k in model_state and model_state[k].shape == v.shape:
            mapped[k] = v

    print(f"매칭된 pretrained tensor 수: {len(mapped)} / {len(model_state)}")
    return mapped


if USE_PRETRAINED:
    print("Pretrained weight 로드 경로:", PRETRAINED_PATH)
    print("Pretrained 출처:", PRETRAINED_SOURCE)
    pre = torch.load(PRETRAINED_PATH, map_location="cpu")
    missing, unexpected = model.load_state_dict(remap(pre), strict=False)
    print("Missing keys:", len(missing))
    print("Unexpected keys:", len(unexpected))
else:
    print("ViT-S/16을 scratch부터 학습")

model = model.to(device)

ViT-S/16을 scratch부터 학습


In [ ]:
epochs = 30
optim = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=5e-2)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)
losses = {a: nn.CrossEntropyLoss() for a in ATTRIBUTES}

logger = WandbLogger(
    project=WANDB_PROJECT,
    run_name=f"level2-vit_s16{'-pretrained' if USE_PRETRAINED else '-scratch'}",
    config={
        "backbone": "vit_s16", "pretrained": USE_PRETRAINED,
        "pretrained_source": PRETRAINED_SOURCE if USE_PRETRAINED else None,
        "epochs": epochs, "batch": BATCH, "lr": 5e-4, "weight_decay": 5e-2, "seed": SEED,
    },
    tags=WANDB_TAGS + ["vit_s16", "pretrained" if USE_PRETRAINED else "scratch"],
)
trainer = MultiTaskTrainer(model, optim, sched, losses, device, TrainConfig(epochs=epochs), logger=logger)

# ViT-S/16 학습 및 리포트 용 history 저장
history = trainer.fit(train_loader, val_loader)

# 학습 종료 후:
val_pred, _, val_tgt, _ = collect_predictions(model, val_loader, device)
cms = confusion_matrices(val_pred, val_tgt)
for a in ATTRIBUTES:
    logger.log_confusion_matrix(f"final/cm_{a}", cms[a], CLASS_NAMES[a])
logger.finish()

os.makedirs("../checkpoints", exist_ok=True)
ckpt_name = f"../checkpoints/level2_vit_s16{'_pretrained' if USE_PRETRAINED else '_scratch'}.pth"
torch.save(
    {
        "state_dict": model.state_dict(),
        "history": history,
        "pretrained": USE_PRETRAINED,
        "pretrained_source": PRETRAINED_SOURCE if USE_PRETRAINED else None,
    },
    ckpt_name,
)
print("saved:", ckpt_name)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


train e1:  95%|█████████▍| 75/79 [00:28<00:01,  2.12it/s, loss=2.1476]

## 분석 (리포트 필수 포함 항목)

1. **CNN vs Transformer**: 동일 epoch 예산 하에서 ResNet-50 (Level 1) 과 ViT-S (Level 2) 의 Avg-MF1 을 비교하세요.
2. **Pretrained vs Scratch**: 약 5천 장 규모의 소규모 데이터셋에서 ImageNet 초기화가 실제로 얼마나 도움이 되는지 정량적으로 보고하세요.
3. **속성별 거동**: ViT 가 ResNet 대비 Weather 와 Time of Day 사이의 오류 분포를 다르게 가져가는지, 그 원인을 가설로 제시하세요.

In [ ]:
# 분석 1, 2) CNN vs Transformer 및 Pretrained vs Scratch 정량 비교
# - Level 1 ResNet-50과 Level 2 ViT-S/16 모두 30 epoch 학습 결과를 비교
# - final_* 컬럼은 마지막 epoch 성능, best_* 컬럼은 학습 중 가장 좋았던 validation 성능

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

CKPT_DIR = Path("../checkpoints")


def load_history_from_ckpt(path):
    if not path.exists():
        print(f"checkpoint 없음: {path}")
        return None
    ckpt = torch.load(path, map_location="cpu")
    if isinstance(ckpt, dict) and "history" in ckpt:
        return ckpt["history"]
    print(f"history가 없는 checkpoint: {path}")
    return None


def summary_from_history(model_name, model_type, pretrained, hist):
    last_epoch = len(hist["val_avg_mf1"]) - 1
    best_epoch = max(range(len(hist["val_avg_mf1"])), key=lambda i: hist["val_avg_mf1"][i])
    last_per = hist["val_per_mf1"][last_epoch]
    best_per = hist["val_per_mf1"][best_epoch]
    return {
        "model": model_name,
        "type": model_type,
        "pretrained": pretrained,
        "epochs_trained": len(hist["val_avg_mf1"]),
        "final_avg_mf1": hist["val_avg_mf1"][last_epoch],
        "final_weather_mf1": last_per["weather"],
        "final_scene_mf1": last_per["scene"],
        "final_timeofday_mf1": last_per["timeofday"],
        "best_epoch": best_epoch + 1,
        "best_avg_mf1": hist["val_avg_mf1"][best_epoch],
        "best_weather_mf1": best_per["weather"],
        "best_scene_mf1": best_per["scene"],
        "best_timeofday_mf1": best_per["timeofday"],
    }


histories = {
    "resnet50_w111": load_history_from_ckpt(CKPT_DIR / "level1_resnet50_w111.pth"),
    "vit_s16_scratch": load_history_from_ckpt(CKPT_DIR / "level2_vit_s16_scratch.pth"),
    "vit_s16_pretrained": load_history_from_ckpt(CKPT_DIR / "level2_vit_s16_pretrained.pth"),
}


# 현재 세션에서 학습한 직후라면 checkpoint가 없어도 history 변수 사용 가능
if "history" in globals():
    histories["vit_s16_pretrained" if USE_PRETRAINED else "vit_s16_scratch"] = history

records = []
if histories["resnet50_w111"] is not None:
    records.append(summary_from_history("ResNet-50", "CNN", False, histories["resnet50_w111"]))
if histories["vit_s16_scratch"] is not None:
    records.append(summary_from_history("ViT-S/16", "Transformer", False, histories["vit_s16_scratch"]))
if histories["vit_s16_pretrained"] is not None:
    records.append(summary_from_history("ViT-S/16", "Transformer", True, histories["vit_s16_pretrained"]))

level2_summary_df = pd.DataFrame(records)
level2_summary_df

In [ ]:
# 분석 1, 2) Validation Avg-MF1 곡선 비교
# - ResNet-50 vs ViT-S/16: CNN 대비 Transformer의 데이터 효율성 확인
# - ViT scratch vs pretrained: ImageNet 초기화가 초반 수렴과 최고 성능에 주는 영향 확인

plt.figure(figsize=(8, 5))

curve_specs = [
    ("ResNet-50", histories.get("resnet50_w111")),
    ("ViT-S/16 scratch", histories.get("vit_s16_scratch")),
    ("ViT-S/16 pretrained", histories.get("vit_s16_pretrained")),
]

for label, hist in curve_specs:
    if hist is not None:
        plt.plot(range(1, len(hist["val_avg_mf1"]) + 1), hist["val_avg_mf1"], label=label)

plt.xlabel("Epoch")
plt.ylabel("Validation Avg-Macro-F1")
plt.title("CNN vs Transformer / Scratch vs Pretrained")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# 분석 3) Weather / Time of Day 오류 분포 비교
# - ResNet-50과 ViT-S/16의 최종 checkpoint 기준 confusion matrix를 비교
# - Transformer가 Weather / Time of Day에서 CNN과 다른 오류 양상을 보이는지 확인

import seaborn as sns
from src.models.resnet import resnet50


def load_model_state(model, path):
    if not path.exists():
        print(f"checkpoint 없음: {path}")
        return None
    ckpt = torch.load(path, map_location="cpu")
    state = ckpt["state_dict"] if isinstance(ckpt, dict) and "state_dict" in ckpt else ckpt
    model.load_state_dict(state, strict=True)
    return model.to(device)


vit_ckpt = CKPT_DIR / "level2_vit_s16_pretrained.pth"
vit_label = "ViT-S/16 pretrained"
if not vit_ckpt.exists():
    vit_ckpt = CKPT_DIR / "level2_vit_s16_scratch.pth"
    vit_label = "ViT-S/16 scratch"

models_for_cm = [
    ("ResNet-50", resnet50(), CKPT_DIR / "level1_resnet50_w111.pth"),
    (vit_label, vit_small_patch16_224(), vit_ckpt),
]

for run_name, run_model, ckpt_path in models_for_cm:
    run_model = load_model_state(run_model, ckpt_path)
    if run_model is None:
        continue

    preds, _, targets, _ = collect_predictions(run_model, val_loader, device)
    cms = confusion_matrices(preds, targets)

    for attr in ["weather", "timeofday"]:
        plt.figure(figsize=(6, 5))
        sns.heatmap(
            cms[attr],
            annot=True,
            fmt=".2f",
            cmap="Blues",
            xticklabels=CLASS_NAMES[attr],
            yticklabels=CLASS_NAMES[attr],
        )
        plt.title(f"{run_name} - {attr} confusion matrix")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.xticks(rotation=45, ha="right")
        plt.yticks(rotation=0)
        plt.tight_layout()
        plt.show()

    run_model.to("cpu")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()